# 04-02. Reading and Building a Four-box Model  
# 4-box モデルの構造とコードを読む・作る

**Ocean Box Modeling with Python / 海洋ボックスモデル入門**

---

## 今日の目的 / Goals

前回の `04-01` では、3-box から 4-box へ進む理由を学びました。  
In `04-01`, we learned why we move from a 3-box model to a 4-box model.

今回は、4-box モデルの中身をより丁寧に見ます。  
In this notebook, we look more carefully inside the 4-box model.

特に、

```text
箱 / boxes
↓
変数 / variables
↓
フラックス / fluxes
↓
保存則 / conservation laws
↓
Python 関数 / Python functions
```

という対応を確認します。  
We check the correspondence between:

```text
boxes
↓
variables
↓
fluxes
↓
conservation laws
↓
Python functions
```

この Notebook のゴールは、完成したプログラムを眺めることではありません。  
The goal is not to simply look at a finished program.

ゴールは、**保存則から Python コードがどう生まれるか**を理解することです。  
The goal is to understand **how Python code emerges from conservation laws**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

plt.rcParams["figure.figsize"] = (7, 4)

## 1. 4-box の再確認 / Review of the four boxes

4-box モデルでは、海洋を次の 4 つの箱に分けます。  
In the 4-box model, the ocean is divided into four boxes.

| Box | 日本語 | English | 主な役割 |
|---|---|---|---|
| H | 南大洋表層 | Southern Ocean surface | 深層水の湧昇・ガス交換 |
| L | 低緯度表層 | Low-latitude surface | 暖かい表層・生物生産 |
| N | 北大西洋表層 | North Atlantic surface | 深層水形成 |
| D | 深層 | Deep ocean | 炭素・栄養塩の貯蔵 |

概念図は次の通りです。  
The conceptual diagram is:

```text
             N
       North Atlantic
          sinking
             ↓

H  →  L  →  N
↑           ↓
└──── D ←───┘

H: Southern Ocean upwelling
L: Low-latitude surface
N: North Atlantic sinking
D: Deep ocean
```

ここでのポイントは、H と N が両方とも高緯度ですが、役割が違うことです。  
The key point is that both H and N are high-latitude boxes, but they have different roles.

In [ ]:
# A compact schematic.
fig, ax = plt.subplots(figsize=(6, 5))

pos = {"H": (0.2, 0.55), "L": (0.5, 0.55), "N": (0.8, 0.55), "D": (0.5, 0.20)}
labels = {
    "H": "H\\nSouthern Ocean\\nupwelling",
    "L": "L\\nLow latitude",
    "N": "N\\nNorth Atlantic\\nsinking",
    "D": "D\\nDeep ocean",
}

for b, (x, y) in pos.items():
    ax.text(x, y, labels[b], ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.5", fc="white", ec="black"))

# Main circulation arrows: D -> H -> L -> N -> D
ax.annotate("", xy=pos["H"], xytext=pos["D"], arrowprops=dict(arrowstyle="->"))
ax.annotate("", xy=pos["L"], xytext=pos["H"], arrowprops=dict(arrowstyle="->"))
ax.annotate("", xy=pos["N"], xytext=pos["L"], arrowprops=dict(arrowstyle="->"))
ax.annotate("", xy=pos["D"], xytext=pos["N"], arrowprops=dict(arrowstyle="->"))

ax.set_xlim(0, 1)
ax.set_ylim(0, 0.9)
ax.axis("off")
ax.set_title("Four-box circulation schematic")
plt.show()

## 2. 変数名のルール / Naming rule for variables

4-box では、同じトレーサーが 4 つの箱にあります。  
In the 4-box model, the same tracer exists in four boxes.

例えば PO4 なら、

```text
PO4H
PO4L
PO4N
PO4D
```

です。

DIC なら、

```text
DICH
DICL
DICN
DICD
```

です。

この命名規則はとても大事です。  
This naming rule is very important.

```text
トレーサー名 + ボックス名
tracer name + box name
```

になっています。  
The name is made as:

```text
tracer name + box name
```

つまり、

```python
DICN
```

は「北大西洋 N box の DIC」です。  
means "DIC in the North Atlantic N box."

In [ ]:
tracers = ["PO4", "DIC", "ALK", "DO2"]
boxes = ["H", "L", "N", "D"]

names = []
for tracer in tracers:
    for box in boxes:
        names.append({"Tracer": tracer, "Box": box, "Variable name": tracer + box})

pd.DataFrame(names)

### Mini exercise 1 / ミニ演習 1

次の変数は何を意味するでしょうか。  
What do the following variables mean?

```text
PO4N
DICH
ALKL
DO2D
```

**答え / Answer**

- `PO4N`: 北大西洋 N box の PO4  
  PO4 in the North Atlantic N box

- `DICH`: 南大洋 H box の DIC  
  DIC in the Southern Ocean H box

- `ALKL`: 低緯度 L box の ALK  
  ALK in the low-latitude L box

- `DO2D`: 深層 D box の O2  
  O2 in the deep box D

## 3. 単位と変換 / Units and conversions

モデル内部では、濃度を `mol/m3` で扱います。  
Inside the model, concentrations are treated in `mol/m3`.

しかし、海洋化学では `umol/kg` の方が見慣れています。  
However, in ocean chemistry, `umol/kg` is more familiar.

そこで変換関数を作ります。  
So we define conversion functions.

```python
to_umolkg(x)
to_ppmv(x)
```

`to_umolkg()` は、モデル内部の `mol/m3` を `umol/kg` に直します。  
`to_umolkg()` converts internal `mol/m3` values to `umol/kg`.

`to_ppmv()` は、pCO2 の atm を ppmv に直します。  
`to_ppmv()` converts pCO2 from atm to ppmv.

In [ ]:
CV1 = 1.0250e3    # mol/kg -> mol/m3
CV2 = 9.7561e-4   # mol/m3 -> mol/kg
CV3 = 1.0e6       # atm -> ppmv
CV4 = 3.1536e7    # seconds/year
CV5 = 8.64e4      # seconds/day

def to_umolkg(x):
    return CV2 * 1.0e6 * x

def to_ppmv(x):
    return CV3 * x

print("1 mol/m3 =", to_umolkg(1.0), "umol/kg")
print("280e-6 atm =", to_ppmv(280e-6), "ppmv")

## 4. ボックス体積 / Box volumes

保存則を書くには、各ボックスの体積が必要です。  
To write conservation laws, we need the volume of each box.

フラックスの単位が `mol/s` のとき、濃度変化は

```text
concentration change = flux * dt / volume
```

です。

したがって、同じフラックスでも、小さいボックスほど濃度変化は大きくなります。  
Therefore, the same flux causes a larger concentration change in a smaller box.

In [ ]:
VOCN_TOTAL = 1.292e18
AOCN = 3.49e14
VATM = 1.773e20

FH_area = 0.10
FN_area = 0.15
FL_area = 0.75

ZH = 250.0
ZN = 250.0
ZL = 100.0

AOCNH = AOCN * FH_area
AOCNN = AOCN * FN_area
AOCNL = AOCN * FL_area

VOCNH = AOCNH * ZH
VOCNN = AOCNN * ZN
VOCNL = AOCNL * ZL
VOCND = VOCN_TOTAL - VOCNH - VOCNN - VOCNL

VOLUME = {"H": VOCNH, "L": VOCNL, "N": VOCNN, "D": VOCND}

pd.DataFrame({
    "Box": ["H", "L", "N", "D"],
    "Volume [m3]": [VOLUME[b] for b in ["H", "L", "N", "D"]],
    "Volume fraction": [VOLUME[b] / VOCN_TOTAL for b in ["H", "L", "N", "D"]],
})

### Mini exercise 2 / ミニ演習 2

同じ `1.0e6 mol/s` のフラックスが H, L, N, D に入ったとします。  
Suppose the same flux of `1.0e6 mol/s` enters H, L, N, and D.

1 日後の濃度変化が最も大きいのはどの箱でしょうか。  
Which box shows the largest concentration change after one day?

実行前に予想してください。  
Predict before running.

In [ ]:
DT = 8.64e4
flux = 1.0e6  # mol/s

rows = []
for b in ["H", "L", "N", "D"]:
    dc = flux * DT / VOLUME[b]
    rows.append({
        "Box": b,
        "Delta concentration [umol/kg]": to_umolkg(dc),
    })

pd.DataFrame(rows)

## 5. 初期値を作る / Create initial values

まず、全ての箱で同じ初期値から始めます。  
First, we start from the same initial values in all boxes.

これは現実的ではありません。  
This is not realistic.

しかし、モデルを走らせると、輸送・生物過程・ガス交換によって箱ごとの差が生まれます。  
However, after running the model, differences among boxes emerge due to transport, biology, and gas exchange.

In [ ]:
TEMP = {"H": 1.0, "L": 21.5, "N": 3.0, "D": 1.75}
SALT = {"H": 34.7, "L": 34.7, "N": 34.7, "D": 34.7}

def initial_four_box():
    x = {}
    for b in ["H", "L", "N", "D"]:
        x[f"PO4{b}"] = 2.10e-6 * CV1
        x[f"DIC{b}"] = 2.235e-3 * CV1
        x[f"ALK{b}"] = 2.374e-3 * CV1
        x[f"DO2{b}"] = 1.60e-4 * CV1
    x["PCO2A"] = 280.0 / CV3
    return x

x = initial_four_box()

pd.DataFrame({
    "Box": ["H", "L", "N", "D"],
    "PO4 [umol/kg]": [to_umolkg(x[f"PO4{b}"]) for b in ["H", "L", "N", "D"]],
    "DIC [umol/kg]": [to_umolkg(x[f"DIC{b}"]) for b in ["H", "L", "N", "D"]],
    "ALK [ueq/kg]": [to_umolkg(x[f"ALK{b}"]) for b in ["H", "L", "N", "D"]],
    "O2 [umol/kg]": [to_umolkg(x[f"DO2{b}"]) for b in ["H", "L", "N", "D"]],
})

## 6. 輸送フラックスを分解して考える / Decomposing transport fluxes

保存則を書くときは、まず各箱について

```text
入るもの
出るもの
```

を分けて考えます。  
When writing conservation laws, first separate what enters and what leaves each box.

ここでは、主循環を

```text
D → H → L → N → D
```

とします。

このとき、PO4 の輸送だけを考えると、

```text
H: D から入る、H から L へ出る
L: H から入る、L から N へ出る
N: L から入る、N から D へ出る
D: N から入る、D から H へ出る
```

です。  
For PO4 transport:

```text
H: receives from D, loses to L
L: receives from H, loses to N
N: receives from L, loses to D
D: receives from N, loses to H
```

In [ ]:
Tcir = 2.0e7  # m3/s

def transport_tendency_PO4_only(x, Tcir=2.0e7):
    tend = {}

    tend["H"] = Tcir * (x["PO4D"] - x["PO4H"])
    tend["L"] = Tcir * (x["PO4H"] - x["PO4L"])
    tend["N"] = Tcir * (x["PO4L"] - x["PO4N"])
    tend["D"] = Tcir * (x["PO4N"] - x["PO4D"])

    return tend

tend = transport_tendency_PO4_only(x, Tcir=Tcir)
pd.DataFrame({
    "Box": ["H", "L", "N", "D"],
    "Transport tendency [mol/s]": [tend[b] for b in ["H", "L", "N", "D"]],
})

初期値が全部同じなので、輸送による変化は 0 です。  
Because all initial values are the same, the transport tendencies are zero.

これは大事です。  
This is important.

濃度差がないと、交換しても濃度は変わりません。  
If there is no concentration difference, exchange does not change concentration.

### Mini exercise 3 / ミニ演習 3

D box の PO4 だけを少し大きくしたら、輸送フラックスはどう変わるでしょうか。  
If only PO4 in the D box is increased, how do transport fluxes change?

予想してから実行してください。  
Predict before running.

In [ ]:
x_test = initial_four_box()
x_test["PO4D"] = 2.50e-6 * CV1

tend = transport_tendency_PO4_only(x_test, Tcir=Tcir)

pd.DataFrame({
    "Box": ["H", "L", "N", "D"],
    "PO4 [umol/kg]": [to_umolkg(x_test[f"PO4{b}"]) for b in ["H", "L", "N", "D"]],
    "Transport tendency [mol/s]": [tend[b] for b in ["H", "L", "N", "D"]],
})

## 7. 生物ポンプを加える / Add the biological pump

生物ポンプは表層ボックスで PO4 と DIC を減らし、深層 D で増やします。  
The biological pump removes PO4 and DIC from surface boxes and adds them to the deep box D.

ここでは、H, L, N で輸出生産が起こるとします。  
Here, export production occurs in H, L, and N.

```text
surface boxes:
  - export

deep box:
  + remineralization
```

```text
表層:
  - 輸出生産

深層:
  + 再無機化
```

In [ ]:
RCP = 106.0
RNP = 16.0
RRC = 0.20
RO2P = 172.0
CV4 = 3.1536e7

CEPH = 0.75
CEPN = 0.02
LF = 0.5
R = 1.0 / CV4
HSC = 2.0e-8 * CV1
DEL = 100.0

def compute_exports(x, CEPH=0.75, CEPN=0.02, LF=0.5):
    EPH = (CEPH / RCP) * AOCNH / CV4
    EPN = (CEPN / RCP) * AOCNN / CV4
    EPL = R * DEL * LF * x["PO4L"] * (x["PO4L"] / (HSC + x["PO4L"])) * VOCNL
    return {"H": EPH, "L": EPL, "N": EPN}

exports = compute_exports(x)

pd.DataFrame({
    "Box": ["H", "L", "N"],
    "Export [mol P/s]": [exports[b] for b in ["H", "L", "N"]],
    "Export [PgC/yr]": [exports[b] * RCP * 12.0e-15 * CV4 for b in ["H", "L", "N"]],
})

### Mini exercise 4 / ミニ演習 4

`LF` を大きくすると、L box の輸出生産はどう変わるでしょうか。  
If `LF` is increased, how does export production in the L box change?

これは 3-box の `EPL` と同じ考え方です。  
This is the same idea as `EPL` in the 3-box model.

In [ ]:
for lf in [0.2, 0.5, 1.0, 2.0]:
    exp = compute_exports(x, LF=lf)
    print(f"LF = {lf:3.1f}, EPL = {exp['L'] * RCP * 12.0e-15 * CV4:.3f} PgC/yr")

## 8. PO4 の保存則を組み立てる / Build the PO4 conservation equations

ここまでで、

```text
輸送
生物ポンプ
体積
時間刻み
```

が揃いました。  
Now we have:

```text
transport
biological pump
volume
time step
```

したがって、PO4 の 1 ステップ更新を書けます。  
Therefore, we can write the one-step update for PO4.

基本形は：

```text
new = old + (transport + biology) * dt / volume
```

です。  
The basic form is:

```text
new = old + (transport + biology) * dt / volume
```

In [ ]:
DT = 8.64e4
FHD = 6.0e7

def update_PO4_one_step(x, Tcir=2.0e7, FHD=6.0e7, LF=0.5):
    y = dict(x)
    exports = compute_exports(x, LF=LF)

    # H receives from D by main circulation and H-D exchange, loses by export
    y["PO4H"] = x["PO4H"] + (
        Tcir * (x["PO4D"] - x["PO4H"])
        + FHD * (x["PO4D"] - x["PO4H"])
        - exports["H"]
    ) * (DT / VOLUME["H"])

    # L receives from H, loses by export
    y["PO4L"] = x["PO4L"] + (
        Tcir * (x["PO4H"] - x["PO4L"])
        - exports["L"]
    ) * (DT / VOLUME["L"])

    # N receives from L, loses by export
    y["PO4N"] = x["PO4N"] + (
        Tcir * (x["PO4L"] - x["PO4N"])
        - exports["N"]
    ) * (DT / VOLUME["N"])

    # D receives from N and H-D exchange, gains remineralized export
    y["PO4D"] = x["PO4D"] + (
        Tcir * (x["PO4N"] - x["PO4D"])
        + FHD * (x["PO4H"] - x["PO4D"])
        + exports["H"] + exports["L"] + exports["N"]
    ) * (DT / VOLUME["D"])

    return y

y = update_PO4_one_step(x)

pd.DataFrame({
    "Box": ["H", "L", "N", "D"],
    "Old PO4 [umol/kg]": [to_umolkg(x[f"PO4{b}"]) for b in ["H", "L", "N", "D"]],
    "New PO4 [umol/kg]": [to_umolkg(y[f"PO4{b}"]) for b in ["H", "L", "N", "D"]],
})

## 9. DIC, ALK, O2 へ拡張する / Extend to DIC, ALK, and O2

PO4 の式が分かれば、DIC, ALK, O2 も同じ考え方で書けます。  
Once we understand the PO4 equation, DIC, ALK, and O2 follow the same logic.

違うのは、生物過程にかかる係数です。  
The difference is the coefficient multiplying biological processes.

```text
PO4: factor = 1
DIC: factor = (1 + RRC) * RCP
ALK: factor = 2 * RRC * RCP - RNP
O2 : factor = RO2P, but sign is opposite for remineralization
```

このように、**構造は同じで係数が違う**と考えると理解しやすくなります。  
It becomes easier if we think: **the structure is the same, but the coefficients differ**.

In [ ]:
alk_factor = 2.0 * RRC * RCP - RNP
dic_factor = (1.0 + RRC) * RCP

pd.DataFrame({
    "Tracer": ["PO4", "DIC", "ALK", "O2"],
    "Biological factor": [1.0, dic_factor, alk_factor, RO2P],
    "Meaning": [
        "phosphorus unit",
        "carbon per phosphorus including CaCO3",
        "alkalinity effect",
        "oxygen per phosphorus",
    ],
})

## 10. 炭酸系とガス交換 / Carbonate chemistry and gas exchange

DIC だけは、大気海洋 CO2 交換も入ります。  
Only DIC also includes air-sea CO2 exchange.

大気と接している箱は、H, L, N です。  
The boxes in contact with the atmosphere are H, L, and N.

```text
H ↔ atmosphere
L ↔ atmosphere
N ↔ atmosphere
D: no direct atmosphere
```

ガス交換の形は簡単化して、

```text
gas flux = piston velocity * solubility * (pCO2_atm - pCO2_ocean)
```

とします。  
We use a simplified gas exchange form:

```text
gas flux = piston velocity * solubility * (pCO2_atm - pCO2_ocean)
```

海洋 pCO2 が大気より低ければ、CO2 は大気から海洋へ入ります。  
If ocean pCO2 is lower than atmospheric pCO2, CO2 enters the ocean.

In [ ]:
def o2sat(TEM, SAL):
    N1 = -1.734292e2
    N2 = +2.496339e2
    N3 = +1.433483e2
    N4 = -2.184920e1
    N5 = -3.309600e-2
    N6 = +1.425900e-2
    N7 = -1.700000e-3
    ATEM = TEM + 273.15
    O2S = math.exp(
        N1 + N2 * 1.0e2 / ATEM
        + N3 * math.log(ATEM / 1.0e2)
        + N4 * ATEM / 1.0e2
        + SAL * (N5 + N6 * ATEM / 1.0e2 + N7 * (ATEM / 1.0e2) ** 2)
    )
    return O2S * 4.35e1 * 1.025e-3

def chemeq_const(TEM, SAL):
    ATEM = TEM + 273.15
    TK = ATEM * 1.0e-2
    SK = 2.3517e-2 + (-2.3656e-2 + 4.7036e-3 * TK) * TK
    K0 = math.exp(-6.02409e1 + 9.34517e1 / TK + 2.33585e1 * math.log(TK) + SK * SAL)
    K1 = math.exp(math.log(10.0) * (
        13.7201 - 3.1334e-2 * ATEM - 3.23576e3 / ATEM
        - 1.3e-5 * SAL * ATEM + 1.032e-1 * math.sqrt(SAL)
    ))
    safe_sal = SAL if SAL > 0 else 1.0
    K2 = math.exp(math.log(10.0) * (
        -5.3719645e3 - 1.671221e0 * ATEM - 2.2913e-1 * SAL
        - 1.83802e1 * math.log10(safe_sal) + 1.2837528e5 / ATEM
        + 2.1943005e3 * math.log10(ATEM) + 8.0944e-4 * SAL * ATEM
        + 5.61711e3 * math.log10(safe_sal) / ATEM - 2.136e0 * SAL / ATEM
    ))
    KB = math.exp(math.log(10.0) * (-9.26 + 8.86e-3 * SAL + 1e-3 * TEM))
    KW = math.exp(
        +1.489802e2 - 1.384726e4 / ATEM - 2.36521e1 * math.log(ATEM)
        + ((-7.92447e1 + 3.29872e3 / ATEM + 1.20408e1 * math.log(ATEM))
           * math.sqrt(SAL) - 1.9813e-2 * SAL)
    )
    BT = 4.106e-4 * SAL / 35.0
    return BT, K0, K1, K2, KB, KW

def co2_nibun(BT, K0, K1, K2, KB, KW, AT, CT):
    ATX = CV2 * AT
    CTX = CV2 * CT
    h_low, h_high = 1.0e-14, 1.0
    hx = 0.5 * (h_low + h_high)

    for _ in range(100000):
        h = 0.5 * (h_low + h_high)
        denom = h*h + K1*h + K1*K2
        at_calc = (
            (2.0*K1*K2*CTX)/denom
            + (h*K1*CTX)/denom
            + (KB*BT)/(h+KB)
            + KW/h
            - h
        )
        if abs(ATX - at_calc) <= 1.0e-15:
            hx = h
            break
        if at_calc < ATX:
            h_high = h
        else:
            h_low = h

    denom2 = hx*hx + K1*hx + K1*K2
    PCO2 = (hx*hx*CTX) / denom2 / K0
    CO2 = (hx*hx*CTX) / denom2 / CV2
    HCO3 = K1 * CO2 / hx
    CO32 = K2 * HCO3 / hx
    return CO2, HCO3, CO32, PCO2

def carbonate_box(temp, sal, alk, dic):
    BT, K0, K1, K2, KB, KW = chemeq_const(temp, sal)
    CO2, HCO3, CO32, PCO2 = co2_nibun(BT, K0, K1, K2, KB, KW, alk, dic)
    return {"K0": K0, "CO2": CO2, "HCO3": HCO3, "CO32": CO32, "PCO2": PCO2}

# Gas exchange parameters
AOCN = 3.49e14
AOCNH = AOCN * 0.10
AOCNN = AOCN * 0.15
AOCNL = AOCN * 0.75

PVH = PVL = PVN = 3.0
FA = {
    "H": PVH * AOCNH / CV5,
    "L": PVL * AOCNL / CV5,
    "N": PVN * AOCNN / CV5,
}

SALT = {"H": 34.7, "L": 34.7, "N": 34.7, "D": 34.7}

rows = []
for b in ["H", "L", "N"]:
    c = carbonate_box(TEMP[b], SALT[b], x[f"ALK{b}"], x[f"DIC{b}"])
    rows.append({
        "Box": b,
        "pCO2_ocean [ppmv]": to_ppmv(c["PCO2"]),
        "pCO2_atm [ppmv]": to_ppmv(x["PCO2A"]),
        "gas exchange coefficient": FA[b],
    })

pd.DataFrame(rows)

## 11. 完全な one-step 関数 / Complete one-step function

ここで、4-box モデルの 1 ステップ関数をまとめます。  
Now we assemble the full one-step function for the 4-box model.

この関数は長いですが、中身はこれまでの積み重ねです。  
The function is long, but it is just a combination of what we have already built.

```text
1. 輸出生産を計算
2. 炭酸系を計算
3. PO4, ALK, DIC を更新
4. O2 を更新
5. 大気 pCO2 を更新
```

```text
1. compute export production
2. compute carbonate chemistry
3. update PO4, ALK, DIC
4. update O2
5. update atmospheric pCO2
```

In [ ]:
VATM = 1.773e20

def one_step_four_box(x, Tcir=2.0e7, FHD=6.0e7, LF=0.5, CEPH=0.75, CEPN=0.02, air_sea=True):
    y = dict(x)

    exports = compute_exports(x, CEPH=CEPH, CEPN=CEPN, LF=LF)
    EPH, EPL, EPN = exports["H"], exports["L"], exports["N"]

    alk_factor = 2.0 * RRC * RCP - RNP
    dic_factor = (1.0 + RRC) * RCP

    # Gas exchange for DIC
    gas = {"H": 0.0, "L": 0.0, "N": 0.0}
    if air_sea:
        for b in ["H", "L", "N"]:
            c = carbonate_box(TEMP[b], SALT[b], x[f"ALK{b}"], x[f"DIC{b}"])
            gas[b] = FA[b] * CV1 * c["K0"] * (x["PCO2A"] - c["PCO2"])

    def update_tracer(prefix, bio_factor, gas_terms=None):
        if gas_terms is None:
            gas_terms = {"H": 0.0, "L": 0.0, "N": 0.0}

        y[f"{prefix}H"] = x[f"{prefix}H"] + (
            Tcir * (x[f"{prefix}D"] - x[f"{prefix}H"])
            + FHD * (x[f"{prefix}D"] - x[f"{prefix}H"])
            - bio_factor * EPH
            + gas_terms.get("H", 0.0)
        ) * (DT / VOLUME["H"])

        y[f"{prefix}L"] = x[f"{prefix}L"] + (
            Tcir * (x[f"{prefix}H"] - x[f"{prefix}L"])
            - bio_factor * EPL
            + gas_terms.get("L", 0.0)
        ) * (DT / VOLUME["L"])

        y[f"{prefix}N"] = x[f"{prefix}N"] + (
            Tcir * (x[f"{prefix}L"] - x[f"{prefix}N"])
            - bio_factor * EPN
            + gas_terms.get("N", 0.0)
        ) * (DT / VOLUME["N"])

        y[f"{prefix}D"] = x[f"{prefix}D"] + (
            Tcir * (x[f"{prefix}N"] - x[f"{prefix}D"])
            + FHD * (x[f"{prefix}H"] - x[f"{prefix}D"])
            + bio_factor * (EPH + EPL + EPN)
        ) * (DT / VOLUME["D"])

    update_tracer("PO4", 1.0)
    update_tracer("ALK", alk_factor)
    update_tracer("DIC", dic_factor, gas_terms=gas)

    # O2
    O2SAT = {b: o2sat(TEMP[b], SALT[b]) for b in ["H", "L", "N", "D"]}

    y["DO2H"] = x["DO2H"] + (
        Tcir * (O2SAT["D"] - x["DO2H"])
        + FHD * (O2SAT["H"] - x["DO2H"])
        + RO2P * EPH
        + FA["H"] * (O2SAT["H"] - x["DO2H"])
    ) * (DT / VOLUME["H"])

    y["DO2L"] = x["DO2L"] + (
        Tcir * (x["DO2H"] - x["DO2L"])
        + RO2P * EPL
        + FA["L"] * (O2SAT["L"] - x["DO2L"])
    ) * (DT / VOLUME["L"])

    y["DO2N"] = x["DO2N"] + (
        Tcir * (x["DO2L"] - x["DO2N"])
        + RO2P * EPN
        + FA["N"] * (O2SAT["N"] - x["DO2N"])
    ) * (DT / VOLUME["N"])

    y["DO2D"] = x["DO2D"] + (
        Tcir * (x["DO2N"] - x["DO2D"])
        + FHD * (x["DO2H"] - x["DO2D"])
        - RO2P * (EPH + EPL + EPN)
    ) * (DT / VOLUME["D"])

    # Atmosphere update
    if air_sea:
        flux_to_atm = 0.0
        for b in ["H", "L", "N"]:
            cnew = carbonate_box(TEMP[b], SALT[b], y[f"ALK{b}"], y[f"DIC{b}"])
            flux_to_atm += FA[b] * CV1 * cnew["K0"] * (cnew["PCO2"] - x["PCO2A"])
        y["PCO2A"] = x["PCO2A"] + flux_to_atm * (DT / VATM)

    diagnostics = {"EPH": EPH, "EPL": EPL, "EPN": EPN}
    return y, diagnostics

## 12. 関数を 1 回だけ試す / Test the function once

長時間積分の前に、1 回だけ実行して値が変わるか確認します。  
Before long-time integration, run the function once and check that values change.

これはデバッグの基本です。  
This is a basic debugging step.

大きなモデルでも、まず 1 ステップだけ動かすのが大事です。  
Even for large models, it is important to first run only one step.

In [ ]:
x = initial_four_box()
y, diag = one_step_four_box(x)

pd.DataFrame({
    "Variable": ["PO4H", "PO4L", "PO4N", "PO4D", "DICH", "DICL", "DICN", "DICD", "PCO2A"],
    "Before": [
        to_umolkg(x["PO4H"]), to_umolkg(x["PO4L"]), to_umolkg(x["PO4N"]), to_umolkg(x["PO4D"]),
        to_umolkg(x["DICH"]), to_umolkg(x["DICL"]), to_umolkg(x["DICN"]), to_umolkg(x["DICD"]),
        to_ppmv(x["PCO2A"]),
    ],
    "After one day": [
        to_umolkg(y["PO4H"]), to_umolkg(y["PO4L"]), to_umolkg(y["PO4N"]), to_umolkg(y["PO4D"]),
        to_umolkg(y["DICH"]), to_umolkg(y["DICL"]), to_umolkg(y["DICN"]), to_umolkg(y["DICD"]),
        to_ppmv(y["PCO2A"]),
    ],
})

## 13. 長時間積分関数 / Long integration function

次に、1 ステップ関数を何度も呼び出す関数を作ります。  
Next, we define a function that repeatedly calls the one-step function.

この構造は 00〜03 と全く同じです。  
This structure is exactly the same as in notebooks 00–03.

```text
初期値を作る
↓
for 文で回す
↓
1 年ごとに保存
↓
DataFrame にする
```

```text
create initial state
↓
run with for-loop
↓
save once per year
↓
make a DataFrame
```

In [ ]:
def diagnose_four_box(x, diag=None):
    row = {}
    for b in ["H", "L", "N", "D"]:
        c = carbonate_box(TEMP[b], SALT[b], x[f"ALK{b}"], x[f"DIC{b}"])
        row[f"PO4{b}"] = to_umolkg(x[f"PO4{b}"])
        row[f"DIC{b}"] = to_umolkg(x[f"DIC{b}"])
        row[f"ALK{b}"] = to_umolkg(x[f"ALK{b}"])
        row[f"DO2{b}"] = to_umolkg(x[f"DO2{b}"])
        row[f"PCO2{b}"] = to_ppmv(c["PCO2"])
    row["PCO2A"] = to_ppmv(x["PCO2A"])
    if diag is not None:
        row["EPH_PgCyr"] = diag["EPH"] * RCP * 12.0e-15 * CV4
        row["EPL_PgCyr"] = diag["EPL"] * RCP * 12.0e-15 * CV4
        row["EPN_PgCyr"] = diag["EPN"] * RCP * 12.0e-15 * CV4
    return row

def run_four_box(years=3000, Tcir=2.0e7, FHD=6.0e7, LF=0.5, CEPH=0.75, CEPN=0.02, air_sea=True):
    x = initial_four_box()
    history = []

    for day in range(years * 365 + 1):
        if day % 365 == 0:
            _, diag = one_step_four_box(x, Tcir=Tcir, FHD=FHD, LF=LF, CEPH=CEPH, CEPN=CEPN, air_sea=air_sea)
            row = {"year": day / 365}
            row.update(diagnose_four_box(x, diag=diag))
            history.append(row)

        x, diag = one_step_four_box(x, Tcir=Tcir, FHD=FHD, LF=LF, CEPH=CEPH, CEPN=CEPN, air_sea=air_sea)

    return x, pd.DataFrame(history)

final, hist = run_four_box(years=3000)
hist.tail()

## 14. 図で確認する / Plot and check

ここでは、4-box モデルが箱ごとの差を作ることを確認します。  
Here we check that the 4-box model creates differences among boxes.

特に、

```text
PO4
DIC
O2
pCO2
```

を見ます。  
We look at PO4, DIC, O2, and pCO2.

In [ ]:
def plot_four_boxes(hist, var, ylabel):
    plt.figure()
    for b in ["H", "L", "N", "D"]:
        plt.plot(hist["year"], hist[f"{var}{b}"], label=b)
    plt.xlabel("Time [yr]")
    plt.ylabel(ylabel)
    plt.title(var)
    plt.legend()
    plt.grid(True)
    plt.show()

plot_four_boxes(hist, "PO4", "PO4 [umol/kg]")
plot_four_boxes(hist, "DIC", "DIC [umol/kg]")
plot_four_boxes(hist, "DO2", "O2 [umol/kg]")

In [ ]:
plt.figure()
plt.plot(hist["year"], hist["PCO2A"], label="Atmosphere")
for b in ["H", "L", "N", "D"]:
    plt.plot(hist["year"], hist[f"PCO2{b}"], label=b)

plt.xlabel("Time [yr]")
plt.ylabel("pCO2 [ppmv]")
plt.title("Atmosphere and ocean pCO2")
plt.legend()
plt.grid(True)
plt.show()

## 15. Challenge A: ガス交換を止める / Turn off air-sea gas exchange

`air_sea=False` にすると、大気海洋 CO2 交換を止めます。  
If `air_sea=False`, air-sea CO2 exchange is turned off.

この場合、大気 pCO2 は変わらないはずです。  
In that case, atmospheric pCO2 should not change.

**予想 / Prediction**

海洋内部の DIC 分布は変わるでしょうか。  
Will the internal ocean DIC distribution still change?

大気 pCO2 は変わるでしょうか。  
Will atmospheric pCO2 change?

In [ ]:
final_noair, hist_noair = run_four_box(years=3000, air_sea=False)

plt.figure()
plt.plot(hist["year"], hist["PCO2A"], label="air_sea=True")
plt.plot(hist_noair["year"], hist_noair["PCO2A"], label="air_sea=False")
plt.xlabel("Time [yr]")
plt.ylabel("Atmospheric pCO2 [ppmv]")
plt.title("Effect of air-sea CO2 exchange")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame({
    "Case": ["air_sea=True", "air_sea=False"],
    "Final pCO2A": [hist["PCO2A"].iloc[-1], hist_noair["PCO2A"].iloc[-1]],
    "Final DICD": [hist["DICD"].iloc[-1], hist_noair["DICD"].iloc[-1]],
})

## 16. Challenge B: 生物ポンプを止める / Turn off the biological pump

次に、輸出生産を弱くします。  
Next, we weaken export production.

ここでは `CEPH=0`, `LF=0`, `CEPN=0` に近い状態を作ります。  
Here we make a case close to `CEPH=0`, `LF=0`, and `CEPN=0`.

**予想 / Prediction**

生物ポンプを止めると、PO4 と DIC の鉛直差は大きくなるでしょうか、小さくなるでしょうか。  
If the biological pump is turned off, do vertical differences in PO4 and DIC become larger or smaller?

In [ ]:
final_nobio, hist_nobio = run_four_box(years=3000, CEPH=0.0, LF=0.0, CEPN=0.0)

plt.figure()
plt.plot(hist["year"], hist["PO4D"] - hist["PO4L"], label="standard")
plt.plot(hist_nobio["year"], hist_nobio["PO4D"] - hist_nobio["PO4L"], label="no biology")
plt.xlabel("Time [yr]")
plt.ylabel("PO4D - PO4L [umol/kg]")
plt.title("Biological pump and vertical nutrient contrast")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame({
    "Case": ["standard", "no biology"],
    "Final PO4D-PO4L": [
        hist["PO4D"].iloc[-1] - hist["PO4L"].iloc[-1],
        hist_nobio["PO4D"].iloc[-1] - hist_nobio["PO4L"].iloc[-1],
    ],
    "Final DICD-DICL": [
        hist["DICD"].iloc[-1] - hist["DICL"].iloc[-1],
        hist_nobio["DICD"].iloc[-1] - hist_nobio["DICL"].iloc[-1],
    ],
})

## 17. Challenge C: コードの一部を読む / Read part of the code

次の部分を見てください。  
Look at this part:

```python
y["PO4L"] = x["PO4L"] + (
    Tcir * (x["PO4H"] - x["PO4L"])
    - exports["L"]
) * (DT / VOLUME["L"])
```

これは L box の PO4 保存則です。  
This is the PO4 conservation law for the L box.

意味は：

```text
現在の PO4L
+ H からの輸送
- L での輸出生産
```

です。  
It means:

```text
current PO4L
+ transport from H
- export production in L
```

### Mini exercise 5 / ミニ演習 5

N box の PO4 式を探して、同じように意味を説明してください。  
Find the PO4 equation for the N box and explain its meaning in the same way.

## 18. 自分で 1 行を改造する / Modify one line yourself

モデルを学ぶ最も良い方法は、1 行だけ変えることです。  
One of the best ways to learn a model is to change only one line.

例えば、L box の輸出生産を 2 倍にしたい場合、  
For example, if we want to double export production in the L box,

```python
exports["L"]
```

を

```python
2.0 * exports["L"]
```

に変えることができます。

ただし、いきなり関数全体を書き換えるのではなく、まずはパラメタ `LF` を変えるのが安全です。  
However, instead of rewriting the whole function immediately, it is safer to change the parameter `LF` first.

In [ ]:
LF_list = [0.25, 0.5, 1.0, 2.0]

summary = []
plt.figure()

for lf in LF_list:
    _, h = run_four_box(years=3000, LF=lf)
    plt.plot(h["year"], h["EPL_PgCyr"], label=f"LF={lf}")
    summary.append({
        "LF": lf,
        "Final EPL": h["EPL_PgCyr"].iloc[-1],
        "Final PO4L": h["PO4L"].iloc[-1],
        "Final pCO2A": h["PCO2A"].iloc[-1],
    })

plt.xlabel("Time [yr]")
plt.ylabel("EPL [PgC/yr]")
plt.title("Changing low-latitude export through LF")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame(summary)

## 19. 課題 / Exercises

### 課題 1 / Exercise 1

4-box モデルの変数名のルールを説明せよ。  
Explain the naming rule for variables in the 4-box model.

### 課題 2 / Exercise 2

L box の PO4 保存則を言葉で説明せよ。  
Explain the PO4 conservation law for the L box in words.

### 課題 3 / Exercise 3

D box の PO4 保存則では、なぜ `EPH + EPL + EPN` が足されるのか。  
Why is `EPH + EPL + EPN` added in the PO4 conservation equation for the D box?

### 課題 4 / Exercise 4

`air_sea=False` にすると、大気 pCO2 と海洋内部のトレーサーはどう変わるか。  
What happens to atmospheric pCO2 and internal ocean tracers when `air_sea=False`?

### 課題 5 / Exercise 5

生物ポンプを止めると、PO4 と DIC の鉛直差はどう変わるか。  
What happens to vertical differences in PO4 and DIC when the biological pump is turned off?

---

## 想定解答 / Expected answers

### 課題 1

変数名は「トレーサー名 + ボックス名」で作られる。例えば `DICN` は N box の DIC、`PO4D` は D box の PO4 を表す。  
Variable names are formed as "tracer name + box name." For example, `DICN` means DIC in the N box, and `PO4D` means PO4 in the D box.

### 課題 2

L box の PO4 は、H box からの輸送で増え、L box の生物輸出で減る。  
PO4 in the L box increases by transport from H and decreases by biological export in L.

### 課題 3

H, L, N の表層で輸出された有機物が深層 D で再無機化されるため、D box では `EPH + EPL + EPN` が足される。  
Organic matter exported from H, L, and N is remineralized in the deep box D, so `EPH + EPL + EPN` is added in D.

### 課題 4

`air_sea=False` では、大気 pCO2 は更新されない。一方、海洋内部では輸送と生物ポンプが残るため、PO4, DIC, O2 の分布は変化しうる。  
When `air_sea=False`, atmospheric pCO2 is not updated. However, internal ocean tracers such as PO4, DIC, and O2 can still change due to transport and biology.

### 課題 5

生物ポンプを止めると、表層から深層への物質輸送が弱くなるため、PO4 と DIC の鉛直差は小さくなる。  
When the biological pump is turned off, transfer of material from surface to deep ocean weakens, so vertical differences in PO4 and DIC become smaller.

## 20. 次へ / Next step

この Notebook では、4-box モデルの中身を分解して読みました。  
In this notebook, we decomposed and read the internal structure of the 4-box model.

次は、感度実験に集中します。  
Next, we focus on sensitivity experiments.

```text
04-03:
  AMOC-like circulation
  Southern Ocean ventilation
  export production
  atmospheric CO2
```

つまり、モデルを「読む」段階から、モデルで「実験する」段階へ進みます。  
We move from reading the model to experimenting with the model.